# A股年报增长率分析

## 项目概述
本 notebook 用于分析2025年年报中增长率大于50%的A股股票。

### 分析目标
- 筛选高增长股票
- 可视化分析结果
- 生成投资建议

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体和图形样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

print("环境配置完成")

In [ ]:
# 加载数据
data_dir = Path('./data')
output_dir = Path('./output')

# 检查数据文件是否存在
results_file = output_dir / 'high_growth_stocks_2025.xlsx'

if results_file.exists():
    # 读取Excel文件
    df = pd.read_excel(results_file, sheet_name='高增长股票')
    print(f"成功加载数据，共 {len(df)} 条记录")
else:
    print("未找到数据文件，请先生成数据")
    df = None

In [ ]:
# 数据概览
if df is not None:
    print("=== 数据概览 ===")
    print(f"总记录数: {len(df)}")
    print(f"列名: {list(df.columns)}")
    
    # 显示前几行数据
    print("\n=== 前5行数据 ===")
    print(df.head())
    
    # 基本统计信息
    print("\n=== 基本统计信息 ===")
    numeric_columns = df.select_dtypes(include=[np.number]).columns
    print(df[numeric_columns].describe())

In [ ]:
# 增长率分布分析
if df is not None and len(df) > 0:
    growth_columns = [
        '营业收入同比增长率',
        '净利润同比增长率', 
        '净资产收益率同比增长率'
    ]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('高增长股票增长率分布', fontsize=16)
    
    for i, col in enumerate(growth_columns):
        if col in df.columns:
            # 过滤异常值（大于1000%）
            filtered_data = df[df[col] <= 10]
            
            # 创建直方图
            axes[i].hist(filtered_data[col], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
            axes[i].set_title(f'{col}分布')
            axes[i].set_xlabel('增长率 (%)')
            axes[i].set_ylabel('股票数量')
            
            # 添加均值线
            mean_val = filtered_data[col].mean()
            axes[i].axvline(mean_val, color='red', linestyle='--', alpha=0.8)
            axes[i].text(0.02, 0.95, f'均值: {mean_val:.1f}%', transform=axes[i].transAxes,
                        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 行业分布分析
if df is not None and '所属行业' in df.columns:
    industry_counts = df['所属行业'].value_counts().head(10)
    
    plt.figure(figsize=(12, 8))
    colors = plt.cm.Set3(np.linspace(0, 1, len(industry_counts)))
    
    bars = plt.barh(industry_counts.index, industry_counts.values, color=colors)
    plt.title('高增长股票行业分布 Top 10', fontsize=16, pad=20)
    plt.xlabel('股票数量')
    plt.ylabel('行业')
    
    # 在每个柱子上添加数值标签
    for i, bar in enumerate(bars):
        width = bar.get_width()
        plt.text(width + max(industry_counts.values)*0.01, 
                bar.get_y() + bar.get_height()/2,
                f'{int(width)}', ha='left', va='center')
    
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# 相关性分析
if df is not None and len(df) > 0:
    growth_cols = [col for col in df.columns if '增长率' in col]
    
    if len(growth_cols) >= 2:
        # 选择相关系数矩阵
        corr_matrix = df[growth_cols].corr()
        
        plt.figure(figsize=(10, 8))
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        
        heatmap = sns.heatmap(corr_matrix, 
                             mask=mask,
                             annot=True, 
                             cmap='coolwarm',
                             center=0,
                             square=True,
                             fmt='.2f',
                             cbar_kws={'shrink': 0.8})
        
        plt.title('财务指标增长率相关性分析', fontsize=16, pad=20)
        plt.tight_layout()
        plt.show()

In [ ]:
# 生成分析报告
if df is not None and len(df) > 0:
    print("=== 高增长股票分析报告 ===")
    print(f"分析时间: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"报告年份: 2025年")
    print(f"筛选条件: 增长率 > 50%")
    print(f"发现高增长股票: {len(df)} 只")
    
    # 按增长率排序
    top_revenue_growth = df.nlargest(5, '营业收入同比增长率')[['stock_code', 'stock_name', '营业收入同比增长率']]
    top_profit_growth = df.nlargest(5, '净利润同比增长率')[['stock_code', 'stock_name', '净利润同比增长率']]
    
    print("\n=== 营收增长Top 5 ===")
    for _, row in top_revenue_growth.iterrows():
        print(f"{row['stock_code']} ({row['stock_name']}): {row['营业收入同比增长率']:.1f}%")
    
    print("\n=== 利润增长Top 5 ===")
    for _, row in top_profit_growth.iterrows():
        print(f"{row['stock_code']} ({row['stock_name']}): {row['净利润同比增长率']:.1f}%")
    
    # 行业分析
    if '所属行业' in df.columns:
    
        industry_performance = df.groupby('所属行业').agg({
            '营业收入同比增长率': 'mean',
            '净利润同比增长率': 'mean',
            'count': 'size'
        }).sort_values('count', ascending=False).head(5)
        
        print("\n=== 表现最佳行业 ===")
        for idx, row in industry_performance.iterrows():
            print(f"{idx}: 平均营收增长 {row['营业收入同比增长率']:.1f}%, "
                  f"平均利润增长 {row['净利润同比增长率']:.1f}%, "
                  f"股票数量 {row['count']}只")